# Data Parallelism: A Tiny 2-Layer ANN, Two Full Copies (CPU + GPU)

**My setup:** Intel i5 (CPU) + NVIDIA GTX 1650 (1 CUDA GPU).

Real Data Parallelism usually means "N identical GPUs, N copies of the model." You only have
one CUDA GPU, so this notebook treats your **CPU and GPU as the two workers** instead — each
gets a **full copy** of the same tiny model, each processes a **different slice of the batch**,
and after each step we manually **average their gradients** (a tiny hand-written version of the
All-Reduce step from `10-all-reduce-and-network.md`) so both copies stay perfectly in sync.

This is the same idea as file `02-data-parallelism.md`, just with 2 workers instead of 4, and CPU
standing in for a second GPU.

> If you get a second CUDA GPU later, change `device_worker1` and `device_worker2` below to
> `cuda:0` and `cuda:1` — nothing else changes.


In [2]:
import copy
import torch
import torch.nn as nn
import torch.optim as optim

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))


PyTorch version: 2.5.1+cu121
CUDA available: True
GPU name: NVIDIA GeForce GTX 1650


## Step 1: Pick the Two Worker Devices

**Keywords:** `worker` = a device holding one full copy of the model, processing its own slice
of data (this is just what we call each "copy" in data parallelism).


In [3]:
device_worker1 = torch.device("cpu")
device_worker2 = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")

print("Worker 1 (replica 1) runs on:", device_worker1)
print("Worker 2 (replica 2) runs on:", device_worker2)


Worker 1 (replica 1) runs on: cpu
Worker 2 (replica 2) runs on: cuda:0


## Step 2: Define the Tiny 2-Layer ANN

Same small model as the Model Parallelism notebook (4 -> 8 -> 1), but this time the **whole
model** goes on each device — nothing is split by layer. What gets split is the **data**, not
the model.


In [4]:
class TinyANN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(4, 8),
            nn.ReLU(),
            nn.Linear(8, 1)
        )

    def forward(self, x):
        return self.net(x)

# Create ONE model first, so both copies start with IDENTICAL weights
base_model = TinyANN()
print(base_model)


TinyANN(
  (net): Sequential(
    (0): Linear(in_features=4, out_features=8, bias=True)
    (1): ReLU()
    (2): Linear(in_features=8, out_features=1, bias=True)
  )
)


## Step 3: Make Two Identical Copies, One Per Worker

Both copies must start with the exact same weights, otherwise they are not true replicas.
We use `copy.deepcopy` on the CPU version, then move one copy to each device.


In [5]:
model_worker1 = copy.deepcopy(base_model).to(device_worker1)
model_worker2 = copy.deepcopy(base_model).to(device_worker2)

# Sanity check: confirm both copies start identical
w1 = model_worker1.net[0].weight.data.cpu()
w2 = model_worker2.net[0].weight.data.cpu()
print("Copies start identical:", torch.allclose(w1, w2))


Copies start identical: True


## Step 4: Dummy Data, Split Into Two Slices

One full batch, split in half — one half goes to Worker 1, the other half to Worker 2.
This is exactly the "Full Batch -> Slice 1 / Slice 2" step from the docs.


In [6]:
torch.manual_seed(0)

num_samples = 256
X = torch.randn(num_samples, 4)
true_weights = torch.tensor([2.0, -1.0, 0.5, 3.0])
y = (X @ true_weights).unsqueeze(1) + 0.1 * torch.randn(num_samples, 1)

half = num_samples // 2
X_slice1, y_slice1 = X[:half], y[:half]      # Worker 1's slice
X_slice2, y_slice2 = X[half:], y[half:]      # Worker 2's slice

print("Worker 1 slice:", X_slice1.shape)
print("Worker 2 slice:", X_slice2.shape)


Worker 1 slice: torch.Size([128, 4])
Worker 2 slice: torch.Size([128, 4])


## Step 5: The Manual "All-Reduce" Helper

After each worker computes its own gradients independently, we need to **average matching
gradients across both copies**, then give both copies the same averaged gradient before the
optimizer step. This function does that averaging — it's a tiny, simplified stand-in for what
NCCL's Ring All-Reduce does automatically across real GPUs (see `10-all-reduce-and-network.md`).


In [7]:
def average_gradients(model_a, model_b):
    """Average matching gradients between two model replicas, in place."""
    for param_a, param_b in zip(model_a.parameters(), model_b.parameters()):
        grad_a = param_a.grad.data
        grad_b = param_b.grad.data.to(grad_a.device)   # bring to a common device to average

        averaged = (grad_a + grad_b) / 2.0

        param_a.grad.data.copy_(averaged.to(param_a.grad.device))
        param_b.grad.data.copy_(averaged.to(param_b.grad.device))


## Step 6: Training Loop

Each step:

1. Worker 1 does a full forward + backward pass on its slice (on CPU).
2. Worker 2 does a full forward + backward pass on its slice (on GPU), independently and at the
   "same time" (in real multi-GPU setups these truly run simultaneously; here they run one after
   another since it's the same Python process, but the *logic* is identical).
3. Gradients are averaged across both copies (`average_gradients`).
4. Both copies apply the same averaged gradient with their own optimizer, so they stay identical.


In [8]:
criterion = nn.MSELoss()
optimizer1 = optim.SGD(model_worker1.parameters(), lr=0.01)
optimizer2 = optim.SGD(model_worker2.parameters(), lr=0.01)

epochs = 200
for epoch in range(epochs):
    optimizer1.zero_grad()
    optimizer2.zero_grad()

    # --- Worker 1: forward + backward on its slice (CPU) ---
    pred1 = model_worker1(X_slice1.to(device_worker1))
    loss1 = criterion(pred1, y_slice1.to(device_worker1))
    loss1.backward()

    # --- Worker 2: forward + backward on its slice (GPU) ---
    pred2 = model_worker2(X_slice2.to(device_worker2))
    loss2 = criterion(pred2, y_slice2.to(device_worker2))
    loss2.backward()

    # --- All-Reduce step: average gradients across both copies ---
    average_gradients(model_worker1, model_worker2)

    # --- Both copies update with the SAME averaged gradient ---
    optimizer1.step()
    optimizer2.step()

    if (epoch + 1) % 20 == 0:
        avg_loss = (loss1.item() + loss2.item()) / 2
        print(f"Epoch {epoch+1:3d}/{epochs} | Worker1 loss: {loss1.item():.4f} "
              f"| Worker2 loss: {loss2.item():.4f} | Avg loss: {avg_loss:.4f}")


Epoch  20/200 | Worker1 loss: 11.7799 | Worker2 loss: 10.7077 | Avg loss: 11.2438
Epoch  40/200 | Worker1 loss: 6.6171 | Worker2 loss: 6.6061 | Avg loss: 6.6116
Epoch  60/200 | Worker1 loss: 1.7221 | Worker2 loss: 1.9117 | Avg loss: 1.8169
Epoch  80/200 | Worker1 loss: 0.5255 | Worker2 loss: 0.6059 | Avg loss: 0.5657
Epoch 100/200 | Worker1 loss: 0.3543 | Worker2 loss: 0.4054 | Avg loss: 0.3798
Epoch 120/200 | Worker1 loss: 0.2852 | Worker2 loss: 0.3273 | Avg loss: 0.3063
Epoch 140/200 | Worker1 loss: 0.2476 | Worker2 loss: 0.2791 | Avg loss: 0.2634
Epoch 160/200 | Worker1 loss: 0.2276 | Worker2 loss: 0.2446 | Avg loss: 0.2361
Epoch 180/200 | Worker1 loss: 0.2169 | Worker2 loss: 0.2220 | Avg loss: 0.2195
Epoch 200/200 | Worker1 loss: 0.2107 | Worker2 loss: 0.2064 | Avg loss: 0.2085


## Step 7: Confirm Both Copies Stayed in Sync

Because both copies received the exact same averaged gradient at every step, their weights
should still match, even though they trained on different data slices.


In [9]:
w1_final = model_worker1.net[0].weight.data.cpu()
w2_final = model_worker2.net[0].weight.data.cpu()

print("Copies still identical after training:", torch.allclose(w1_final, w2_final, atol=1e-6))


Copies still identical after training: True


## Step 8: Quick Sanity Check

Either copy should now give the same prediction, since they're identical.


In [10]:
model_worker1.eval()
model_worker2.eval()

with torch.no_grad():
    sample = torch.tensor([[1.0, 0.5, -0.2, 2.0]])
    pred_from_worker1 = model_worker1(sample.to(device_worker1))
    pred_from_worker2 = model_worker2(sample.to(device_worker2))

    print("Prediction from Worker 1 (CPU):", pred_from_worker1.item())
    print("Prediction from Worker 2 (GPU):", pred_from_worker2.item())


Prediction from Worker 1 (CPU): 6.743462562561035
Prediction from Worker 2 (GPU): 6.743462562561035


## What Just Happened (Recap)

1. Both `model_worker1` and `model_worker2` started as **identical full copies** of the same
   tiny model.
2. Each step, each worker computed its **own gradients**, using its **own slice** of the batch,
   completely independently.
3. We averaged the two workers' gradients (`average_gradients`) — a hand-rolled version of
   **All-Reduce**.
4. Both workers applied the **same averaged gradient**, so they stayed identical the entire time
   — this is exactly the Data Parallelism guarantee from the docs.

**In a real multi-GPU setup**, you would not write `average_gradients` by hand — PyTorch's
`torch.nn.parallel.DistributedDataParallel` (DDP) does this automatically and efficiently using
NCCL's Ring All-Reduce (see `10-all-reduce-and-network.md`), across as many real GPUs as you have.

### If You Get a Second CUDA GPU Later

Just change:

```python
device_worker1 = torch.device("cuda:0")
device_worker2 = torch.device("cuda:1")
```

Nothing else in this notebook needs to change.
